# Suite RND — Random sources

A store is equipped with a random source when it is made, for whatever draws from it. `Stores.Random` is the protocol
and `Stores.PCG32` the reference source, specified exactly, so that both implementations draw the same numbers.

In [ ]:
import { Bindings as Bd, Proxies, Schemas as S, Stores } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, conformance_problems, raises, same, text } from "./support.js";

const REFERENCE = [0xa15c02b7n, 0x7b47f409n, 0xba1d3330n, 0x83d2f293n, 0xbfa4784bn, 0xcbed606en]; // pcg32-demo: seed 42, sequence 54

## RND-01 · PCG32 draws the reference sequence

In [ ]:
{
  const source = new Stores.PCG32(42n, 54n);
  assert(same([...Array(6)].map(() => source.next_u32()), REFERENCE) && source.seed === 42n && source.sequence === 54n);
  assert(new Stores.PCG32(42n).sequence === 0n && new Stores.PCG32(2n ** 64n - 1n, 2n ** 64n - 1n).next_u32() < 2n ** 32n);
  for (const [seed, sequence] of [[-1n, 0n], [2n ** 64n, 0n], [0n, -1n], [0n, 2n ** 64n], [1.0, 0n], [0n, "1"]] as const) {
    raises(ValueError, () => new Stores.PCG32(seed as never, sequence as never), "a PCG32's seed and sequence are ints from 0 to 2**64 - 1");
  }
}

## RND-02 · A split stream is determined by the seed and the key alone

In [ ]:
{
  const source = new Stores.PCG32(42n);
  const contact = source.split("Contact");
  assert(contact.seed === 11264215962035836729n && contact.sequence === 0n); // FNV-1a 64 of the key, from the basis XOR the seed
  assert(same([contact.next_u32(), contact.next_u32(), contact.next_u32()], [2216387846n, 723670704n, 428412945n]));
  source.next_u32();
  assert(source.split("Contact").next_u32() === 2216387846n); // what was drawn before does not matter
  assert(new Stores.PCG32(42n, 7n).split("Contact").sequence === 7n && new Stores.PCG32(43n).split("Contact").seed !== contact.seed);
  assert(source.split("Contact").split("0").split("age").seed === contact.split("0").split("age").seed);
  assert(source.split("Ünïcödé ✓").seed !== source.split("Unicode").seed); // by the key's UTF-8 bytes
  assert(source.split("\u{1F600}\u00e9\u0800").seed !== source.split("").seed); // every width of UTF-8
  raises(ValueError, () => source.split("\ud800"), "a key must be text without lone surrogates");
}

## RND-03 · A store is equipped with a source when it is made

In [ ]:
{
  const source = new Stores.PCG32(7n);
  assert(new Proxies.OfStore({ random: source }).random() === source);
  const Item = new S.OfObject.Builder().name("Item").ref().properties(text("name")).create();
  const bound = new Bd.OfStore([[Item, () => null]], [], { random: source });
  assert(bound.random() === source);
  raises(LookupError, () => new Proxies.OfStore().random(), "the store has no random source: give it one when it is made");
  assert(typeof source.next_u32 === "function" && typeof source.split === "function");
}